# REPASO II

In [1]:
# Estructuras de Datos Globales
INVENTARIO_LIBROS = [] 
USUARIOS = {}
MULTAS_COBRADAS = 0.0
ID_PRESTAMO_COUNTER = 1 

print("Estructuras de datos globales inicializadas.")
print(f"Libros: {INVENTARIO_LIBROS}")
print(f"Usuarios: {USUARIOS}")

Estructuras de datos globales inicializadas.
Libros: []
Usuarios: {}


In [3]:
def agregar_libro(isbn, titulo, autor, cantidad_total):
    """Agrega un libro al inventario o aumenta su stock si ya existe."""
    # Buscar libro existente
    for libro in INVENTARIO_LIBROS:
        if libro['isbn'] == isbn:
            libro['cantidad_total'] += cantidad_total
            return f"Stock de '{titulo}' actualizado. Nuevo total: {libro['cantidad_total']}."
    
    # Agregar nuevo libro
    nuevo_libro = {
        'isbn': isbn,
        'titulo': titulo,
        'autor': autor,
        'cantidad_total': cantidad_total,
        'cantidad_prestada': 0
    }
    INVENTARIO_LIBROS.append(nuevo_libro)
    return f"Libro '{titulo}' agregado al inventario."


In [5]:
def eliminar_libro(isbn):
    """Elimina un libro si no tiene copias prestadas."""    
    for libro in INVENTARIO_LIBROS:
        if libro['isbn'] == isbn:
            if libro['cantidad_prestada'] == 0:
                INVENTARIO_LIBROS.remove(libro)
                return f"Libro con ISBN {isbn} eliminado correctamente."
            else:
                return f"ERROR: No se puede eliminar el libro. Hay {libro['cantidad_prestada']} copias prestadas."
    return f"ERROR: Libro con ISBN {isbn} no encontrado."



In [6]:

def obtener_inventario():
    """Retorna una lista de libros con su disponibilidad."""
    inventario_disponible = []
    for libro in INVENTARIO_LIBROS:
        copias_disponibles = libro['cantidad_total'] - libro['cantidad_prestada']
        inventario_disponible.append({
            'titulo': libro['titulo'],
            'autor': libro['autor'],
            'copias_disponibles': copias_disponibles
        })
    return inventario_disponible


In [7]:
def buscar_libro_por_titulo(titulo_buscado):
    """Busca libros por coincidencia parcial en el título."""
    resultados = []
    titulo_buscado_lower = titulo_buscado.lower()
    for libro in INVENTARIO_LIBROS:
        if titulo_buscado_lower in libro['titulo'].lower():
            resultados.append(libro)
    return resultados

In [8]:
def agregar_usuario(id_usuario, nombre, email):
    if id_usuario in USUARIOS:
        return f"ERROR: El ID de usuario {id_usuario} ya existe."
    
    USUARIOS[id_usuario] = {
        'nombre': nombre,
        'email': email,
        'prestamos_activos': []
    }
    return f"Usuario '{nombre}' agregado con éxito."


In [9]:

def obtener_usuarios(con_prestamos=False):
    """Retorna la lista de usuarios, opcionalmente incluyendo el número de préstamos activos."""
    info_usuarios = {}
    for id_usuario, data in USUARIOS.items():
        info_usuarios[id_usuario] = {
            'nombre': data['nombre'],
            'email': data['email']
        }
        if con_prestamos:
            info_usuarios[id_usuario]['prestamos_activos_count'] = len(data['prestamos_activos'])
            
    return info_usuarios

In [11]:
def _obtener_libro(isbn):
    """Función auxiliar para encontrar un libro por ISBN."""
    for libro in INVENTARIO_LIBROS:
        if libro['isbn'] == isbn:
            return libro
    return None

def realizar_prestamo(id_usuario, isbn):
    """Registra un préstamo de un libro a un usuario."""
    global ID_PRESTAMO_COUNTER
    
    # 1. Verificar Usuario
    if id_usuario not in USUARIOS:
        return f"ERROR: Usuario con ID {id_usuario} no encontrado."

    # 2. Verificar Libro
    libro = _obtener_libro(isbn)
    if not libro:
        return f"ERROR: Libro con ISBN {isbn} no encontrado."

    # 3. Verificar Disponibilidad
    copias_disponibles = libro['cantidad_total'] - libro['cantidad_prestada']
    if copias_disponibles <= 0:
        return f"ERROR: No hay copias disponibles de '{libro['titulo']}'."
    
    # 4. Registrar Préstamo
    
    # Actualizar Libro
    libro['cantidad_prestada'] += 1
    
    # Actualizar Usuario
    USUARIOS[id_usuario]['prestamos_activos'].append(isbn)
    
    # Generar ID (Opcional, pero bueno para la trazabilidad)
    prestamo_id = ID_PRESTAMO_COUNTER
    ID_PRESTAMO_COUNTER += 1
    
    return f"Préstamo (ID: {prestamo_id}) de '{libro['titulo']}' a '{USUARIOS[id_usuario]['nombre']}' registrado con éxito."

def registrar_devolucion(id_usuario, isbn):
    """Registra la devolución de un libro."""
    # 1. Verificar Usuario
    if id_usuario not in USUARIOS:
        return f"ERROR: Usuario con ID {id_usuario} no encontrado."

    # 2. Verificar si el libro está prestado al usuario
    if isbn not in USUARIOS[id_usuario]['prestamos_activos']:
        return f"ERROR: El usuario {USUARIOS[id_usuario]['nombre']} no tiene prestado el libro con ISBN {isbn}."
        
    # 3. Actualizar Usuario (Eliminar de activos)
    USUARIOS[id_usuario]['prestamos_activos'].remove(isbn)
    
    # 4. Actualizar Libro (Disminuir prestados)
    libro = _obtener_libro(isbn)
    if libro:
        libro['cantidad_prestada'] -= 1
        return f"Devolución de '{libro['titulo']}' registrada con éxito."
    else:
        # Esto no debería pasar si el ISBN existía en el usuario
        return "ADVERTENCIA: Libro devuelto, pero no se encontró en el inventario."

In [13]:
def calcular_prestamos_activos_totales():
    """Calcula la suma de todas las copias prestadas actualmente."""
    total_activos = sum(libro['cantidad_prestada'] for libro in INVENTARIO_LIBROS)
    return total_activos

def obtener_usuarios_con_mas_prestamos(n=3):
    """Identifica a los N usuarios con más préstamos activos."""
    
    # Crear una lista de tuplas (nombre, cantidad_prestamos)
    rankings = []
    for id_usuario, data in USUARIOS.items():
        cantidad = len(data['prestamos_activos'])
        rankings.append((data['nombre'], cantidad))
    
    # Ordenar de forma descendente por la cantidad de préstamos (índice 1)
    rankings.sort(key=lambda x: x[1], reverse=True)
    
    # Retornar los N primeros
    return rankings[:n]

def registrar_multa(monto):
    """Aumenta el contador global de multas cobradas."""
    global MULTAS_COBRADAS
    MULTAS_COBRADAS += monto
    return f"Multa de {monto:.2f} registrada. Total multas: {MULTAS_COBRADAS:.2f}."

In [14]:
# --- Pruebas de Libros ---
print("--- PRUEBAS DE LIBROS ---")
print(agregar_libro("978-123", "Cien años de soledad", "G. G. Márquez", 10))
print(agregar_libro("978-456", "El Quijote", "M. de Cervantes", 5))
print(agregar_libro("978-123", "Cien años de soledad", "G. G. Márquez", 2)) # Aumentar stock

print("\n--- PRUEBAS DE USUARIOS ---")
# --- Pruebas de Usuarios ---
print(agregar_usuario("U001", "Ana Pérez", "ana@mail.com"))
print(agregar_usuario("U002", "Luis García", "luis@mail.com"))

print("\n--- PRUEBAS DE PRÉSTAMOS ---")
# --- Pruebas de Préstamos ---
print(realizar_prestamo("U001", "978-123")) # Éxito
print(realizar_prestamo("U001", "978-123")) # Éxito
print(realizar_prestamo("U002", "978-456")) # Éxito

print("\n--- ESTADÍSTICAS Y MULTAS ---")
# --- Pruebas de Estadísticas y Multas ---
print("Inventario disponible:")
print(obtener_inventario())

print(f"Total de préstamos activos: {calcular_prestamos_activos_totales()}")

print(registrar_multa(15.50))

print("\n--- PRUEBAS DE DEVOLUCIONES ---")
# --- Pruebas de Devoluciones ---
print(registrar_devolucion("U001", "978-123")) # Éxito
print(f"Total de préstamos activos después de devolución: {calcular_prestamos_activos_totales()}")

print("\n--- RANKING ---")
print("Top 3 Usuarios con más préstamos:")
print(obtener_usuarios_con_mas_prestamos(3))

--- PRUEBAS DE LIBROS ---
Stock de 'Cien años de soledad' actualizado. Nuevo total: 22.
Stock de 'El Quijote' actualizado. Nuevo total: 10.
Stock de 'Cien años de soledad' actualizado. Nuevo total: 24.

--- PRUEBAS DE USUARIOS ---
ERROR: El ID de usuario U001 ya existe.
ERROR: El ID de usuario U002 ya existe.

--- PRUEBAS DE PRÉSTAMOS ---
Préstamo (ID: 4) de 'Cien años de soledad' a 'Ana Pérez' registrado con éxito.
Préstamo (ID: 5) de 'Cien años de soledad' a 'Ana Pérez' registrado con éxito.
Préstamo (ID: 6) de 'El Quijote' a 'Luis García' registrado con éxito.

--- ESTADÍSTICAS Y MULTAS ---
Inventario disponible:
[{'titulo': 'Cien años de soledad', 'autor': 'G. G. Márquez', 'copias_disponibles': 20}, {'titulo': 'El Quijote', 'autor': 'M. de Cervantes', 'copias_disponibles': 8}]
Total de préstamos activos: 6
Multa de 15.50 registrada. Total multas: 15.50.

--- PRUEBAS DE DEVOLUCIONES ---
Devolución de 'Cien años de soledad' registrada con éxito.
Total de préstamos activos después de 